[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1337-Artificial-Intelligence/HackAI-Challenges-2026/blob/main/notebooks/agents/01_llm_agents.ipynb)


# Before You Start

## What is an LLM Agent?

> ### **Common LLM Limitations:**
1. **Hallucination**
   - LLMs can generate incorrect information with high confidence
<center>
<img src="https://i.ibb.co/xSCvT7HM/agent-pr1.png" alt="agent pr1" border="0">
</center>

>2. **Knowledge Cutoff**
   - LLMs are limited to information from their training data
   - They cannot access or learn from information after their training period

<center>
<img src="https://i.ibb.co/3YQgC50m/agent-pr2.png" alt="agent pr2" border="0">
</center>

>3. **Data Privacy**
   - LLMs can only access public training data
   - They cannot access proprietary or private information

<center>
<img src="https://i.ibb.co/qZX4Dkf/agent-pr3.png" alt="agent pr3" border="0">
</center>

> ### **The Solution: LLM Agents**
An LLM Agent enhances a basic LLM by combining three key components:
- **LLM**: The core language model
- **Tools**: External capabilities and functions
- **Memory**: Ability to store and recall information

<center>
<img src="https://i.ibb.co/ycdMJt6r/agent.png" alt="agent" border="0" height="500">
</center>


> This combination helps overcome the limitations by:
- Using tools to verify information and access external data
- Maintaining memory of past interactions and information
- Enabling access to current data through external tools

## How Does it Work?

> The LLM agent follows a structured decision-making process when responding to instructions. This process consists of three main steps that repeat until the task is complete:
>1. **Think/Planning:** The agent analyzes the user's request and formulates a step-by-step plan to accomplish the task.
>2. **Action:** The agent executes specific actions by calling appropriate tools with the necessary parameters.
>3. **Observation:** The agent evaluates the results of its actions and determines the next steps based on the outcomes.

<center>
<img src="https://i.ibb.co/h1YJf6FT/agent2.png" alt="agent2" border="0" height="500">
</center>

>This cycle continues until the agent successfully completes the requested task or determines it cannot proceed further.

# Build Your First Agent Using LangGraph

> Let's build a sample agent that can:
* **Search the internet for information**
* **Do math calculations (+,-,*,/)**
* **Check the weather of a city**
* **arxiv paper info**

<center>
<img src="https://i.ibb.co/Mk5zzv1x/agent3.png" alt="agent3" border="0">
</center>

#### 1. install langchain library

In [2]:
%pip install -U langgraph
# ddgs for duckducksearch engine
%pip install langchain_community ddgs
# groq as llm provider
%pip install langchain_groq
# load API keys from the .env file
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


* set env for agent tracing with [langsmith](https://smith.langchain.com)

In [10]:
import os
from dotenv import load_dotenv

# Load the API keys from the .env file (at the root of the project) into os.environ.
# Fill in your keys in .env, never in the notebook. See .env.example.
# LangChain reads the LANGSMITH_* variables automatically.
load_dotenv()

print("GROQ_API_KEY loaded:", bool(os.getenv("GROQ_API_KEY")))
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))

GROQ_API_KEY loaded: False
LANGSMITH_TRACING: true


#### 2. Select Your LLM Model

<center>
<img src="https://i.ibb.co/k2b4Cv4j/agent4.png" alt="agent4" border="0" height="300">
</center>

>`langchain` supports multiple LLM providers including OpenAI, Google, and HuggingFace. For this tutorial, we'll use OpenAI's OSS `openai/gpt-oss-120b` model through their free API at **Groq**.

>To get started:
> 1. Visit [Groq Console](https://console.groq.com/keys)
2. Create an account if you don't have one
3. Generate an API key
4. Keep your API key secure - we'll use it in the next step.


In [4]:
from langchain_groq import ChatGroq
# load model
MODEL = "openai/gpt-oss-120b"
GROQ_API_KEY = os.getenv("GROQ_API_KEY")  # read from the .env file
model = ChatGroq(
    model=MODEL,
    api_key = GROQ_API_KEY
)

ModuleNotFoundError: No module named 'langchain_groq'

In [ ]:
response = model.invoke(
    [
        {
            "role": "user",
            "content": "What is the capital of Morocco?",
        }
    ]
).content
response

'The capital of Morocco is **Rabat**.'

#### 3. Define Tools

>langchain supports multiple tool types:
- Build in LangChain [tools](https://docs.langchain.com/oss/python/integrations/tools)
- MCP (Model Control Protocol) tools
- Building your own one

>In this example, we'll demonstrate how to:
1. Create custom tools (using `@tool`)
2. Use predefined tools from `langchain`

>**Find out more langchain** [tools](https://docs.langchain.com/oss/python/integrations/tools).

<center>
<img src="https://i.ibb.co/cXrXzBH6/agent5.png" alt="agent5" border="0" height="500">
</center>

In [ ]:
from langchain.tools import tool
@tool
def calculator_tool(expression:str) -> str:
  # the tool should have a clear name. and a string documnetation.
  # string documentation has description of the tool and its inputs.
  "A calculator that can perform basic arithmetic operations"
  try:
      result = eval(expression)
      return str(result)
  except Exception as e:
      return f"Error: {str(e)}"

# test tool
print(calculator_tool.run("2 + 2 * 3"))

8


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
@tool
def city_weather(city: str) -> str:
    # the tool should have a clear name. and a string documnetation.
    # string documentation has description of the tool and its inputs.
    """
    Get the weather for a given city.
    Args:
        city: The name of the city.
    """
    # In a real-world scenario, this function would call a weather API.
    # For this example, we'll just return a dummy response.
    return f"The weather in {city} is sunny with a high of 25°C."

# test city_weather
tool_output = city_weather.run("Benguerir")
print(tool_output)

The weather in Benguerir is sunny with a high of 25°C.


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

# Create a DuckDuckGo search tool
search_tool = DuckDuckGoSearchRun()
# Test Tool
tool_output = search_tool.invoke("What is the capital of Morocco?")
print(tool_output)

Morocco,[c] officially the Kingdom of Morocco,[d] is a country in the Maghreb region of North Africa.The culture of Morocco is a mix of Arab, Berber, European (specifically Andalusian[17]), and African cultures. Its capital is Rabat, while its largest city is Casablanca.[18]. Join me as I travel and vlog in the capital city of Morocco, Rabat. I walk through the beautiful Andalusian Gardens, explore the unbelievable Rabat Kasbah, a... Rabat is the capital and one of Morocco’s four imperial cities and the Moroccan government seat, housing the royal palace, the parliament, and several embassies, underscoring its role in the nation’s governance. The capital of Morocco is Rabat, which is not the most populated city in the country. It is located where the Bou Regreg River meets the Atlantic Ocean. Its population is approximately 577,827; the entire metro area has a population of around 1.2 million. The capital of Morocco is Rabat, located in the North African continent. It serves as the poli

In [ ]:
# arxiv>=3 removed Search.results(), which the langchain_community ArxivQueryRun tool still uses
%pip install "arxiv<3"

Using Python 3.12.13 environment at: /usr
Checked 1 package in 97ms


In [ ]:
from langchain_community.tools.arxiv.tool import ArxivQueryRun

arxiv_tool = ArxivQueryRun()
tool_output = arxiv_tool.run("1706.03762")
print(tool_output)

Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new sing

#### 4. Integrate Tools: Create Your Agent

In [ ]:
from langchain.agents import create_agent

tools=[city_weather,search_tool,calculator_tool,arxiv_tool]
agent=create_agent(
    model=model,
    tools=tools
)

#### 5. Run Your Agent

In [ ]:
query="""What's the paper 1706.03762 about,
and who is Noam Shazeer?
Also, what is the weather in Benguerir?
And what is 2 + 2 * 3?"""
response=agent.invoke(
    {
    "messages": [{"role": "user", "content": query}]}
)

In [ ]:
print(response["messages"][-1].content)

**Paper 1706.03762 – “Attention Is All You Need”**  
*Published:* June 2017 (arXiv:1706.03762)  
*Authors:* Ashish Vaswani, **Noam Shazeer**, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  

**What it’s about**  
The paper introduces the **Transformer** architecture, a novel neural‑network design for sequence‑to‑sequence tasks (e.g., machine translation) that relies **solely on attention mechanisms** and eliminates recurrent or convolutional layers. Key points:

| Aspect | Description |
|--------|-------------|
| **Core idea** | Use multi‑head self‑attention to relate all positions in a sequence to each other in parallel. |
| **Benefits** | - Much higher parallelism (faster training). <br> - State‑of‑the‑art translation quality (BLEU scores > 28 on English‑German, > 41 on English‑French). <br> - Fewer parameters and lower computational cost compared with LSTM/CNN‑based models. |
| **Architecture** | Encoder‑decoder stacks of identical layers

#### 6. Add Memory

- If you want your agent to use memory you should to add short term memory with `InMemorySaver`, and also the `thread_id` is what keeps the conversation memory connected across calls, each new conversation will have a `thread_id`` .

<center>
<img src="https://i.ibb.co/Nd7cxpTC/agent7.png" alt="agent7" border="0" height="300">
</center>

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Short-term memory storage
memory = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[city_weather, search_tool, calculator_tool, arxiv_tool],
    checkpointer=memory
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "user_1" # set your id here
    }
}

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "hello my name abdeljalil"}
        ]
    },
    config=config
)
print(response["messages"][-1].content)

Hello, Abdeljalil! Nice to meet you. How can I help you today?


- by activating the memory the agent remember my name

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "what is my name?"}
        ]
    },
    config=config
)
print(response["messages"][-1].content)

Your name is **Abdeljalil**.


# See Agent Traces

After setting the langsmith variable above, you'll find the traces in [langsmith](https://smith.langchain.com)

* **example:** https://smith.langchain.com/public/ef878f04-9627-44fd-be6c-19a8de83faa7/r

# Exercice

## Exercise plan

We improve the first agent in 4 steps:
1. **A real weather tool**: replace the dummy `city_weather` with real data from the free [Open-Meteo](https://open-meteo.com) API (no API key needed).
2. **A safe calculator**: replace `eval()` (which can run *any* Python code) with a parser that only accepts numbers and arithmetic operators.
3. **A system prompt**: tell the agent how to behave (answer in the user's language, always use tools for weather and math, no tool when not needed).
4. **Tests**: check each tool alone, then the full agent (multi-tool question, no-tool question, memory).

### 1. Real weather tool (Open-Meteo)

Two API calls:
1. **Geocoding**: city name → latitude/longitude.
2. **Forecast**: latitude/longitude → current weather.

The API returns a numeric `weather_code` (WMO standard), which we translate into words with `WEATHER_CODES`.
If the city is not found, the tool returns a hint so that the agent can **retry with another spelling** by itself.

In [ ]:
import requests
from langchain.tools import tool

WEATHER_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "depositing rime fog",
    51: "light drizzle", 53: "moderate drizzle", 55: "dense drizzle",
    61: "light rain", 63: "moderate rain", 65: "heavy rain",
    71: "light snow", 73: "moderate snow", 75: "heavy snow",
    80: "light rain showers", 81: "moderate rain showers", 82: "violent rain showers",
    95: "thunderstorm", 96: "thunderstorm with light hail", 99: "thunderstorm with heavy hail",
}

@tool
def real_city_weather(city: str) -> str:
    """
    Get the current real weather for a given city: conditions, temperature,
    feels-like temperature, humidity, wind, and today's min/max temperature.
    Args:
        city: The name of the city, e.g. "Benguerir" or "Casablanca".
    """
    try:
        # 1. city name -> coordinates (Open-Meteo geocoding API, no key needed)
        # try English then French names (e.g. "Marrakech" is only found in French)
        for language in ("en", "fr"):
            geo = requests.get(
                "https://geocoding-api.open-meteo.com/v1/search",
                params={"name": city, "count": 1, "language": language},
                timeout=10,
            ).json()
            if geo.get("results"):
                break
        if not geo.get("results"):
            return (f"City '{city}' not found. Try another spelling "
                    f"(e.g. with a space: 'Ben Guerir' instead of 'Benguerir').")
        place = geo["results"][0]

        # 2. coordinates -> weather (Open-Meteo forecast API, no key needed)
        data = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,apparent_temperature,relative_humidity_2m,wind_speed_10m,weather_code",
                "daily": "temperature_2m_max,temperature_2m_min",
                "timezone": "auto",
                "forecast_days": 1,
            },
            timeout=10,
        ).json()
        current, daily = data["current"], data["daily"]
        conditions = WEATHER_CODES.get(current["weather_code"], "unknown conditions")

        return (
            f"Weather in {place['name']}, {place.get('country', '')} (at {current['time']} local time): "
            f"{conditions}, {current['temperature_2m']}°C "
            f"(feels like {current['apparent_temperature']}°C), "
            f"humidity {current['relative_humidity_2m']}%, wind {current['wind_speed_10m']} km/h. "
            f"Today: min {daily['temperature_2m_min'][0]}°C, max {daily['temperature_2m_max'][0]}°C."
        )
    except Exception as e:
        return f"Error while fetching the weather: {e}"

In [ ]:
# test the weather tool alone
print(real_city_weather.run("Ben Guerir"))
print(real_city_weather.run("Ifrane"))
# a spelling the API doesn't know -> the tool returns a hint for the agent
print(real_city_weather.run("Benguerir"))

### 2. Safe calculator

`eval()` executes **any** Python code. The LLM decides what to send to the tool, so a malicious prompt could make it run something dangerous.
The cell below shows the problem with the original `calculator_tool` (here it only prints the current folder, harmless):

In [ ]:
# ⚠️ the original calculator runs arbitrary Python code
print(calculator_tool.run("__import__('os').getcwd()"))

The safe version uses `ast` (Python's own parser) to turn the expression into a tree, then only computes **numbers** and **arithmetic operators**.
Anything else (function calls, variables, imports...) is refused. Huge exponents are refused too, so `9 ** 9999999` can't freeze the notebook.

In [ ]:
from langchain.tools import tool
import ast
import operator

_ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def _safe_eval(node):
    # numbers only (bool is a subclass of int, so exclude it)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
        return node.value
    # binary operations: 2 + 3, 4 * 5, ...
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        left, right = _safe_eval(node.left), _safe_eval(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError("exponent too large")
        return _ALLOWED_OPS[type(node.op)](left, right)
    # unary operations: -5, +3
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    # anything else (function calls, variables, imports, ...) is refused
    raise ValueError(f"unsupported element: {type(node).__name__}")

@tool
def safe_calculator(expression: str) -> str:
    """
    A safe calculator for arithmetic expressions.
    Supports + - * / // % ** and parentheses.
    Args:
        expression: the arithmetic expression to compute, e.g. "2 + 2 * 3".
    """
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_safe_eval(tree.body))
    except ZeroDivisionError:
        return "Error: division by zero"
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# test the safe calculator alone
tests = [
    "2 + 2 * 3",                   # 8
    "(10 - 4) / 3",                # 2.0
    "2 ** 10",                     # 1024
    "1 / 0",                       # division by zero
    "__import__('os').getcwd()",   # refused: function call
    "2 ** 1000",                   # refused: exponent too large
]
for expression in tests:
    print(f"{expression!r:30} -> {safe_calculator.run(expression)}")

### 3. Improved agent: new tools + system prompt + memory

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM_PROMPT = """You are a helpful assistant.
- Always answer in the user's language (French, English, Arabic or Moroccan Darija).
  Moroccan Darija is often written in Latin letters with digits (3 = ع, 7 = ح, 9 = ق), e.g. 'chhal kayn dyal l7rara': answer such messages in Darija too, in the same script.
- For any weather question, use the real_city_weather tool. If a city is not found, retry with another spelling.
- For any calculation, use the safe_calculator tool. Never compute in your head.
- Do not call any tool when the question doesn't need one (greetings, general chat)."""

agent_v2 = create_agent(
    model=model,
    tools=[real_city_weather, safe_calculator, search_tool, arxiv_tool],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

### 4. Tests

The helper `ask()` sends a question and prints **which tools the agent called** (with their arguments) before the final answer.

In [ ]:
def ask(agent, question, thread_id="test"):
    config = {"configurable": {"thread_id": thread_id}}
    response = agent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)

    # with memory, response["messages"] holds the whole conversation:
    # keep only the messages produced for this question
    messages = response["messages"]
    last_question = max(i for i, m in enumerate(messages) if m.type == "human")
    tool_calls = [tc for m in messages[last_question:] for tc in getattr(m, "tool_calls", [])]

    print("Question:", question)
    print("Tools called:", [f"{tc['name']}({tc['args']})" for tc in tool_calls] or "none")
    print("Answer:", messages[-1].content)
    print("-" * 80)

In [ ]:
# test 1: several tools in one question (weather twice + calculator)
ask(agent_v2, "What's the weather in Benguerir and in Casablanca? What is the temperature difference between the two cities?")

In [ ]:
# test 2: no tool should be called
ask(agent_v2, "Bonjour, comment vas-tu ?")

In [ ]:
# test 3: answer in the user's language (Darija)
ask(agent_v2, "chhal kayn dyal l7rara f Marrakech daba?")

In [ ]:
# test 4: memory -- same thread_id, the agent remembers
ask(agent_v2, "Hi, my name is Jihane and I live in Rabat.", thread_id="memory_test")
ask(agent_v2, "What's my name, and what's the weather where I live?", thread_id="memory_test")

# new thread_id -> new conversation, the agent doesn't know
ask(agent_v2, "What's my name?", thread_id="another_user")

In [ ]:
# test 5: the calculator refuses code even if the user asks for it
ask(agent_v2, "Use the calculator to compute __import__('os').getcwd()")

In [ ]:
# see every step of the last conversation in detail (LLM messages, tool calls, tool results)
config = {"configurable": {"thread_id": "memory_test"}}
for message in agent_v2.get_state(config).values["messages"]:
    message.pretty_print()